# Protein Druggability Assessment — Software Agent

**Agents:** Structure · Biology · Safety · Chemistry · Decision

> Run cells **top to bottom**. All outputs are saved to `/kaggle/working/`.

In [3]:
!pip install -q requests pandas rdkit biopython scipy scikit-learn py3Dmol numpy

In [11]:
import numpy as np
import os
import requests
import py3Dmol
from Bio.PDB import PDBParser, PDBList
from scipy.spatial import KDTree
from sklearn.cluster import DBSCAN


HYDROPHOBIC = {"ALA","VAL","LEU","ILE","MET","PHE","TRP","PRO"}


class StructureAgent:

    def __init__(self, protein_input):
        self.pdb_file = self.resolve_input(protein_input)
        self.models = []
        self.load_structure()

    # ------------------------------------------------
    # Resolve input (Protein name / PDB ID / File)
    # ------------------------------------------------
    def resolve_input(self, protein_input):

        protein_input = protein_input.strip()

        if os.path.exists(protein_input):
            return protein_input

        clean = protein_input.upper()

        if "PDB" in clean:
            clean = clean.split()[-1]

        if len(clean) == 4 and clean.isalnum():
            return self.download_pdb(clean)

        # Text search fallback
        print("Searching RCSB for:", protein_input)

        query = {
            "query": {
                "type": "terminal",
                "service": "text",
                "parameters": {"value": protein_input}
            },
            "return_type": "entry",
            "request_options": {"pager": {"start": 0, "rows": 1}}
        }

        response = requests.post(
            "https://search.rcsb.org/rcsbsearch/v1/query",
            json=query
        )

        data = response.json()

        if "result_set" not in data or len(data["result_set"]) == 0:
            raise ValueError("No structure found. Try PDB ID directly.")

        pdb_id = data["result_set"][0]["identifier"]
        print("Found PDB:", pdb_id)

        return self.download_pdb(pdb_id)

    def download_pdb(self, pdb_id):

        pdbl = PDBList()
        file_path = pdbl.retrieve_pdb_file(
            pdb_id,
            pdir=".",
            file_format="pdb"
        )

        new_name = f"{pdb_id}.pdb"

        if os.path.exists(new_name):
            return new_name

        os.rename(file_path, new_name)

        return new_name

    # ------------------------------------------------
    # Load Structure
    # ------------------------------------------------
    def load_structure(self):

        parser = PDBParser(QUIET=True)
        structure = parser.get_structure("protein", self.pdb_file)

        for model in structure:

            atoms = []
            residues = []
            res_ids = []

            for chain in model:
                prev_res = None

                for residue in chain:

                    if residue.get_resname() == "HOH":
                        continue

                    res_id = residue.get_id()[1]

                    if prev_res and res_id != prev_res + 1:
                        print(f"GAP detected between {prev_res} and {res_id}")

                    prev_res = res_id

                    for atom in residue:
                        if atom.get_parent().id[0] != " ":
                            continue
                        atoms.append(atom.coord)
                        residues.append(residue.get_resname())
                        res_ids.append(res_id)

            self.models.append({
                "atoms": np.array(atoms),
                "residues": residues,
                "res_ids": res_ids
            })

    # ------------------------------------------------
    # Pocket Detection (Improved)
    # ------------------------------------------------
    def detect_pocket(self, atoms):

        tree = KDTree(atoms)
        spacing = 1.0

        min_c = np.min(atoms, axis=0) - 2
        max_c = np.max(atoms, axis=0) + 2

        x = np.arange(min_c[0], max_c[0], spacing)
        y = np.arange(min_c[1], max_c[1], spacing)
        z = np.arange(min_c[2], max_c[2], spacing)

        grid = np.array(np.meshgrid(x, y, z)).T.reshape(-1, 3)

        cavity = []

        for point in grid:
            dist, _ = tree.query(point)

            if dist > 2.8:
                neighbors = tree.query_ball_point(point, r=5.0)
                if 8 < len(neighbors) < 80:
                    cavity.append(point)

        cavity = np.array(cavity)

        if len(cavity) == 0:
            return None

        clustering = DBSCAN(eps=2.5, min_samples=8).fit(cavity)
        labels = clustering.labels_

        largest = None
        max_size = 0

        for label in set(labels):
            if label == -1:
                continue
            cluster = cavity[labels == label]
            if len(cluster) > max_size:
                largest = cluster
                max_size = len(cluster)

        return largest

    # ------------------------------------------------
    # Extract Pocket Residues
    # ------------------------------------------------
    def extract_pocket_residues(self, atoms, residues, res_ids, pocket):

        tree = KDTree(atoms)
        pocket_res = set()

        for point in pocket:
            neighbors = tree.query_ball_point(point, r=4.0)
            for idx in neighbors:
                pocket_res.add(res_ids[idx])

        return sorted(pocket_res)

    # ------------------------------------------------
    # Evaluate + Visualize
    # ------------------------------------------------
    def evaluate(self):

        print("Total Models:", len(self.models))

        model = self.models[0]

        atoms = model["atoms"]
        residues = model["residues"]
        res_ids = model["res_ids"]

        pocket = self.detect_pocket(atoms)

        if pocket is None:
            print("No druggable pocket detected.")
            return

        pocket_res_ids = self.extract_pocket_residues(
            atoms, residues, res_ids, pocket
        )

        volume = len(pocket)
        depth = np.max(pocket[:,2]) - np.min(pocket[:,2])

        print("Pocket Volume:", volume)
        print("Pocket Depth:", round(depth,2))
        print("Pocket Residue IDs:", pocket_res_ids)

        self.visualize(pocket_res_ids)

    # ------------------------------------------------
    # AlphaFold-style Visualization
    # ------------------------------------------------
    def visualize(self, pocket_res_ids):

        with open(self.pdb_file, "r") as f:
            pdb_data = f.read()

        view = py3Dmol.view(width=900, height=700)
        view.addModel(pdb_data, "pdb")

        # Cartoon structure
        view.setStyle({"cartoon": {"color": "spectrum"}})

        # Highlight pocket residues
        if pocket_res_ids:
            view.setStyle(
                {"resi": pocket_res_ids},
                {"stick": {"colorscheme": "redCarbon"}}
            )

        view.addSurface(py3Dmol.VDW, {"opacity": 0.2})
        view.zoomTo()
        view.spin(True)

        # Save as HTML for Kaggle (no interactive display support)
        html_path = self.pdb_file.replace(".pdb", "_pocket.html")
        view.write_html(html_path)
        print(f"3D visualization saved to: {html_path}")


# ------------------------------------------------
# MAIN
# ------------------------------------------------
if __name__ == "__main__":

    protein_input = input("Enter Protein Name, PDB ID, or File: ")

    agent = StructureAgent(protein_input)
    viewer = agent.evaluate()

Enter Protein Name, PDB ID, or File:  1UWH


GAP detected between 599 and 612
GAP detected between 722 and 1723
GAP detected between 599 and 612
GAP detected between 722 and 1723
Total Models: 1
Pocket Volume: 8487
Pocket Depth: 63.0
Pocket Residue IDs: [447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 512, 513, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 524, 525, 526, 528, 529, 530, 531, 532, 533, 534, 535, 537, 538, 539, 540, 541, 542, 543, 544, 545, 546, 547, 548, 549, 550, 551, 552, 553, 554, 555, 557, 558, 561, 563, 564, 565, 566, 567, 568, 569, 570, 571, 572, 573, 574, 575, 576, 577, 578, 579, 580, 581, 582, 583, 584, 585, 586, 587, 588, 590, 591, 592, 593, 594, 595, 596, 597, 598, 599, 612, 613, 614, 615, 616, 617, 618, 619, 620, 621, 622, 623, 

In [12]:
# -*- coding: utf-8 -*-
"""SE project - bio agent.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1RUUt6BTtyxwYXwrQMI35l09VslnHGJIr
"""

"""
Biology Agent for Protein Druggability Assessment
- Determines if a protein plays a key role in causing disease
- Avoids targets where blocking the protein does not affect disease outcome
"""

import requests
import json
import os

OPEN_TARGETS_URL = "https://api.platform.opentargets.org/api/v4/graphql"

def get_ensembl_id(gene_symbol: str) -> tuple:
    query = """
    query SearchTarget($symbol: String!) {
      search(queryString: $symbol, entityNames: ["target"]) {
        hits {
          id
          name
          entity
          object {
            ... on Target {
              id
              approvedSymbol
              approvedName
            }
          }
        }
      }
    }
    """
    response = requests.post(
        OPEN_TARGETS_URL,
        json={"query": query, "variables": {"symbol": gene_symbol}},
        timeout=15
    )
    data = response.json()
    hits = data.get("data", {}).get("search", {}).get("hits", [])

    for hit in hits:
        obj = hit.get("object", {})
        if obj.get("approvedSymbol", "").upper() == gene_symbol.upper():
            return obj["id"], obj["approvedName"]

    if hits:
        obj = hits[0].get("object", {})
        return obj.get("id"), obj.get("approvedName")

    return None, None

def get_disease_associations(ensembl_id: str) -> list:
    query = """
    query GetAssociations($id: String!) {
      target(ensemblId: $id) {
        associatedDiseases(page: {index: 0, size: 200}) {
          rows {
            score
            disease {
              id
              name
            }
            datatypeScores {
              id
              score
            }
          }
        }
      }
    }
    """
    response = requests.post(
        OPEN_TARGETS_URL,
        json={"query": query, "variables": {"id": ensembl_id}},
        timeout=15
    )
    data = response.json()
    target = data.get("data", {}).get("target", {})
    rows = target.get("associatedDiseases", {}).get("rows", [])
    return rows


# -------------------------------
# FIXED DISEASE MATCH FUNCTION
# -------------------------------
def find_disease_match(associations: list, disease_query: str) -> dict:
    disease_query_lower = disease_query.lower().strip()

    best_match = None
    best_score = 0

    for row in associations:
        disease_name = row["disease"]["name"].lower().strip()

        # Strict match (substring match only)
        if disease_query_lower in disease_name or disease_name in disease_query_lower:

            if row["score"] > best_score:
                best_score = row["score"]
                best_match = row

    return best_match


def parse_evidence_types(datatype_scores: list) -> dict:
    evidence_map = {
        "genetic_association": "Genetic mutations in patients",
        "somatic_mutation":    "Somatic mutations in disease tissue",
        "known_drug":          "Existing drugs target this protein",
        "affected_pathway":    "Protein is in disease pathway",
        "literature":          "Mentioned in research papers",
        "animal_model":        "Animal knockout/overexpression studies",
        "rna_expression":      "Gene expression changes in disease",
        "text_mining":         "Text mining from publications"
    }
    parsed = {}
    for item in datatype_scores:
        label = evidence_map.get(item["id"], item["id"])
        parsed[label] = round(item["score"], 3)
    return parsed


def rule_based_assessment(overall_score: float, evidence: dict) -> dict:
    flags = []
    avoid_flags = []

    if evidence.get("Genetic mutations in patients", 0) > 0.3:
        flags.append("Strong genetic evidence: mutations found in patients")
    if evidence.get("Somatic mutations in disease tissue", 0) > 0.3:
        flags.append("Somatic mutations present - likely driver not bystander")
    if evidence.get("Existing drugs target this protein", 0) > 0.2:
        flags.append("Already a known drug target - druggability validated")
    if evidence.get("Animal knockout/overexpression studies", 0) > 0.2:
        flags.append("Animal model evidence: blocking this protein affects disease")

    if evidence.get("Genetic mutations in patients", 0) < 0.1 and \
       evidence.get("Somatic mutations in disease tissue", 0) < 0.1:
        avoid_flags.append("No genetic/somatic mutation evidence - may be bystander")
    if evidence.get("Gene expression changes in disease", 0) > 0.3 and \
       evidence.get("Genetic mutations in patients", 0) < 0.1:
        avoid_flags.append("Only expression change - likely bystander, not driver")

    if overall_score >= 0.6:
        verdict = "STRONG TARGET"
    elif overall_score >= 0.35:
        verdict = "MODERATE TARGET"
    elif overall_score >= 0.15:
        verdict = "WEAK TARGET - needs more evidence"
    else:
        verdict = "AVOID - insufficient disease link"

    return {
        "verdict": verdict,
        "overall_score": round(overall_score, 3),
        "positive_signals": flags,
        "warning_signals": avoid_flags
    }


def biology_agent(gene_symbol: str, disease_name: str):
    print(f"\n{'='*60}")
    print(f"  BIOLOGY AGENT - Druggability Assessment")
    print(f"{'='*60}")
    print(f"  Protein : {gene_symbol}")
    print(f"  Disease : {disease_name}")
    print(f"{'='*60}\n")

    print("[1/4] Looking up protein in Open Targets...")
    ensembl_id, full_name = get_ensembl_id(gene_symbol)
    if not ensembl_id:
        print(f"ERROR: Could not find '{gene_symbol}' in Open Targets.")
        return
    print(f"      Found: {gene_symbol} ({full_name}) -> {ensembl_id}")

    print("[2/4] Fetching disease associations...")
    associations = get_disease_associations(ensembl_id)
    if not associations:
        print("ERROR: No disease associations found.")
        return
    print(f"      Found {len(associations)} disease associations.")

    print(f"[3/4] Matching to '{disease_name}'...")
    match = find_disease_match(associations, disease_name)

    if not match:
      print(f"      No exact match found for '{disease_name}'.")
      print("      Avoiding incorrect fallback to unrelated disease.")
      print("\nRESULT: LOW BIOLOGICAL RELEVANCE (No valid disease match)")
      return
    matched_disease = match["disease"]["name"]
    overall_score = match["score"]
    print(f"      Best match: '{matched_disease}' (score: {round(overall_score, 3)})")

    print("[4/4] Parsing evidence types...")
    evidence = parse_evidence_types(match.get("datatypeScores", []))
    for k, v in evidence.items():
        print(f"      {k}: {v}")

    assessment = rule_based_assessment(overall_score, evidence)


    print(f"\n{'='*60}")
    print(f"  FINAL REPORT")
    print(f"{'='*60}")
    print(f"  Protein       : {gene_symbol} - {full_name}")
    print(f"  Disease       : {matched_disease}")
    print(f"  Overall Score : {assessment['overall_score']}/1.0")
    print(f"  Verdict       : {assessment['verdict']}")

    if assessment["positive_signals"]:
        print(f"\n  Positive Signals:")
        for s in assessment["positive_signals"]:
            print(f"     + {s}")

    if assessment["warning_signals"]:
        print(f"\n  Warning Signals:")
        for s in assessment["warning_signals"]:
            print(f"     ! {s}")



if __name__ == "__main__":
    biology_agent("BRAF", "melanoma")

    # More to try:
    # biology_agent("EGFR", "breast cancer")
    # biology_agent("BRAF", "melanoma")
    # biology_agent("KRAS", "pancreatic cancer")
    # "GAPDH", "diabetes"


  BIOLOGY AGENT - Druggability Assessment
  Protein : BRAF
  Disease : melanoma

[1/4] Looking up protein in Open Targets...
      Found: BRAF (B-Raf proto-oncogene, serine/threonine kinase) -> ENSG00000157764
[2/4] Fetching disease associations...
      Found 200 disease associations.
[3/4] Matching to 'melanoma'...
      Best match: 'melanoma' (score: 0.819)
[4/4] Parsing evidence types...
      clinical: 0.976
      Protein is in disease pathway: 0.434
      Mentioned in research papers: 0.997
      Genetic mutations in patients: 0.699
      Somatic mutations in disease tissue: 0.796

  FINAL REPORT
  Protein       : BRAF - B-Raf proto-oncogene, serine/threonine kinase
  Disease       : melanoma
  Overall Score : 0.819/1.0
  Verdict       : STRONG TARGET

  Positive Signals:
     + Strong genetic evidence: mutations found in patients
     + Somatic mutations present - likely driver not bystander


In [13]:
"""
SAFETY AGENT - LEVEL 3 (Kaggle Compatible)
============================================
Protein Druggability Assessment System
Team No. 8

What this agent does:
1. Checks where protein is expressed in the body (tissue expression)
2. Flags vital organ risks
3. Checks known side effect databases
4. Compares against historical failed drugs
5. Predicts specific side effects
6. Gives confidence score with full reasoning

FIXES in this version:
- Gene name extraction fixed for UniProt API response
- Historical comparison checks both gene name AND input name
- File paths updated for Kaggle (/kaggle/working/)
- Fallback data expanded for more proteins
- API parsing made more robust
"""

import requests
import json
import time
from dataclasses import dataclass
from typing import Optional


# ─────────────────────────────────────────────
# DATA STRUCTURES
# ─────────────────────────────────────────────

@dataclass
class SafetyReport:
    protein_id: str
    protein_name: str
    tissue_expression: dict
    flagged_organs: list
    known_side_effects: list
    historical_comparison: dict
    predicted_side_effects: list
    risk_score: float
    confidence: float
    verdict: str
    reasoning: str


# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

# Vital organs and their risk weights
# Higher weight = more dangerous if affected
VITAL_ORGANS = {
    "heart":        3.0,
    "brain":        3.0,
    "liver":        2.5,
    "kidney":       2.0,
    "lung":         2.0,
    "pancreas":     1.5,
    "bone marrow":  2.5,
    "spinal cord":  2.5
}

# Expression levels mapped to numbers
EXPRESSION_LEVELS = {
    "High":         3,
    "Medium":       2,
    "Low":          1,
    "Not detected": 0,
    "":             0
}

# Known dangerous protein families (from historical drug failures)
HIGH_RISK_FAMILIES = [
    "herg",   # Heart channel — causes cardiac arrest
    "cyp",    # Liver enzymes — drug interactions
    "nav",    # Sodium channels — neurotoxicity
    "cav",    # Calcium channels — cardiac issues
]

# Historical failed drug targets database
HISTORICAL_FAILED_TARGETS = {
    "KCNH2": {
        "also_known_as": "hERG",
        "failure_reason": "Cardiac arrhythmia — many drugs pulled from market",
        "risk": "HIGH"
    },
    "HERG": {
        "also_known_as": "KCNH2",
        "failure_reason": "Cardiac arrhythmia",
        "risk": "HIGH"
    },
    "CYP3A4": {
        "also_known_as": "Cytochrome P450 3A4",
        "failure_reason": "Severe drug-drug interactions",
        "risk": "HIGH"
    },
    "PTGS1": {
        "also_known_as": "COX-1",
        "failure_reason": "Gastrointestinal bleeding",
        "risk": "MODERATE"
    },
    "COX1": {
        "also_known_as": "PTGS1",
        "failure_reason": "Gastrointestinal bleeding",
        "risk": "MODERATE"
    },
    "SCN5A": {
        "also_known_as": "Nav1.5",
        "failure_reason": "Cardiac sodium channel — arrhythmia risk",
        "risk": "HIGH"
    },
    "CACNA1C": {
        "also_known_as": "Cav1.2",
        "failure_reason": "Cardiac calcium channel — cardiovascular toxicity",
        "risk": "HIGH"
    },
}

# Expanded fallback expression data for well-known proteins
# Used when APIs are unavailable or return bad data
FALLBACK_EXPRESSION = {
    "EGFR": {
        "lung": "High", "liver": "Medium", "kidney": "Low",
        "heart": "Low", "skin": "High", "intestine": "Medium"
    },
    "BRCA1": {
        "breast": "High", "ovary": "High", "liver": "Low",
        "kidney": "Low", "heart": "Low"
    },
    "TP53": {
        "liver": "High", "lung": "High", "brain": "Medium",
        "heart": "Low", "kidney": "Medium"
    },
    "KCNH2": {
        "heart": "High", "brain": "Medium", "liver": "Low"
    },
    "HERG": {
        "heart": "High", "brain": "Medium", "liver": "Low"
    },
    "CYP3A4": {
        "liver": "High", "intestine": "High", "kidney": "Low"
    },
    "PTGS1": {
        "stomach": "High", "kidney": "High", "platelet": "High",
        "heart": "Low", "liver": "Low"
    },
    "BRAF": {
        "lung": "Medium", "liver": "Low", "kidney": "Low",
        "brain": "Low", "skin": "High"
    },
    "HER2": {
        "breast": "High", "heart": "Medium", "liver": "Low",
        "lung": "Low", "kidney": "Low"
    },
    "BCL2": {
        "bone marrow": "High", "liver": "Medium", "heart": "Low",
        "lung": "Low", "kidney": "Low"
    },
    "ALK": {
        "lung": "High", "brain": "Low", "liver": "Low",
        "kidney": "Low", "heart": "Low"
    },
    "VEGFR": {
        "lung": "Medium", "liver": "Medium", "kidney": "High",
        "heart": "Medium", "brain": "Low"
    },
}


# ─────────────────────────────────────────────
# MODULE 1: UNIPROT LOOKUP
# Converts protein name to UniProt ID
# and fetches basic protein info
# ─────────────────────────────────────────────

def lookup_protein(protein_input: str) -> dict:
    """
    Takes a protein name or UniProt ID
    Returns protein details from UniProt
    Falls back gracefully if API fails
    """
    print(f"\n🔍 Looking up protein: {protein_input}")

    protein_input_clean = protein_input.strip()

    try:
        # Search by name/ID
        url = f"https://rest.uniprot.org/uniprotkb/search?query={protein_input_clean}&format=json&size=1"
        response = requests.get(url, timeout=10)
        data = response.json()

        if "results" not in data or not data["results"]:
            raise ValueError("No results found in UniProt")

        entry = data["results"][0]

        # ── Extract protein name ──
        protein_name = protein_input_clean  # fallback
        try:
            protein_name = (
                entry.get("proteinDescription", {})
                     .get("recommendedName", {})
                     .get("fullName", {})
                     .get("value", protein_input_clean)
            )
        except Exception:
            pass

        # ── Extract UniProt accession ID ──
        uniprot_id = entry.get("primaryAccession", protein_input_clean)

        # ── Extract gene name (FIXED) ──
        # UniProt API returns genes as a list of dicts
        gene_name = protein_input_clean  # always fallback to input
        try:
            genes = entry.get("genes", [])
            if genes and isinstance(genes, list):
                first_gene = genes[0]
                if isinstance(first_gene, dict):
                    gene_obj = first_gene.get("geneName", {})
                    if isinstance(gene_obj, dict):
                        extracted = gene_obj.get("value", "")
                        if extracted:
                            gene_name = extracted
        except Exception:
            pass  # keep fallback

        print(f"   ✓ Found: {protein_name} | UniProt: {uniprot_id} | Gene: {gene_name}")

        return {
            "uniprot_id": uniprot_id,
            "protein_name": protein_name,
            "gene_name": gene_name,
        }

    except Exception as e:
        print(f"   ⚠ UniProt lookup failed: {e}")
        print(f"   ℹ Using input directly: {protein_input_clean}")
        return {
            "uniprot_id": protein_input_clean,
            "protein_name": protein_input_clean,
            "gene_name": protein_input_clean,
        }


# ─────────────────────────────────────────────
# MODULE 2: TISSUE EXPRESSION
# Checks where in the body this protein is found
# Uses Human Protein Atlas API with fallback
# ─────────────────────────────────────────────

def get_tissue_expression(gene_name: str, protein_input: str) -> dict:
    """
    Queries Human Protein Atlas for tissue expression data
    Falls back to built-in database if API fails
    Returns dict of {organ: expression_level}
    """
    print(f"\n🧬 Checking tissue expression for: {gene_name}")

    # Try HPA API first
    expression = _try_hpa_api(gene_name)

    # If HPA failed or returned nothing, try with original input
    if not expression and gene_name != protein_input:
        expression = _try_hpa_api(protein_input)

    # If still nothing, use fallback database
    if not expression:
        expression = _get_fallback_expression(gene_name, protein_input)

    return expression


def _try_hpa_api(name: str) -> dict:
    """Attempts to fetch from Human Protein Atlas"""
    try:
        url = f"https://www.proteinatlas.org/{name}/tissue.json"
        response = requests.get(url, timeout=10)

        if response.status_code != 200:
            return {}

        # Check content type — HPA sometimes returns HTML on failure
        content_type = response.headers.get("Content-Type", "")
        if "html" in content_type:
            return {}

        data = response.json()
        expression = {}

        if isinstance(data, list):
            for item in data:
                tissue = item.get("tissue", "").lower().strip()
                level = item.get("level", "Not detected")
                if tissue:
                    expression[tissue] = level

        elif isinstance(data, dict):
            tissues = data.get("tissueExpression", [])
            for item in tissues:
                tissue = item.get("tissue", "").lower().strip()
                level = item.get("level", "Not detected")
                if tissue:
                    expression[tissue] = level

        if expression:
            print(f"   ✓ Found HPA expression data in {len(expression)} tissues")

        return expression

    except Exception as e:
        print(f"   ⚠ HPA API failed: {e}")
        return {}


def _get_fallback_expression(gene_name: str, protein_input: str) -> dict:
    """
    Returns built-in expression data for well-known proteins
    Checks multiple name variants
    """
    print(f"   ℹ Using built-in expression database")

    # Try various name formats
    for key in [gene_name.upper(), protein_input.upper(),
                gene_name, protein_input]:
        if key in FALLBACK_EXPRESSION:
            data = FALLBACK_EXPRESSION[key]
            print(f"   ✓ Found built-in data for: {key}")
            return data

    # Generic minimal fallback
    print(f"   ⚠ No built-in data found — using minimal defaults")
    return {
        "liver": "Low",
        "kidney": "Low",
        "heart": "Low"
    }


# ─────────────────────────────────────────────
# MODULE 3: KNOWN SIDE EFFECTS
# Queries ChEMBL for documented safety liabilities
# ─────────────────────────────────────────────

def get_known_side_effects(gene_name: str, protein_input: str) -> list:
    """
    Queries ChEMBL API for known target safety issues
    Returns list of documented side effects
    """
    print(f"\n⚠️  Checking known side effects for: {gene_name}")

    side_effects = []

    # Try both gene name and original input
    for name in [gene_name, protein_input]:
        if not name or name == gene_name and name != gene_name:
            continue
        result = _query_chembl(name)
        if result:
            side_effects = result
            break

    if not side_effects:
        print(f"   ℹ No documented side effects found in ChEMBL")

    return side_effects


def _query_chembl(name: str) -> list:
    """Queries ChEMBL for a given protein name"""
    try:
        url = f"https://www.ebi.ac.uk/chembl/api/data/target.json?target_synonym={name}&format=json"
        response = requests.get(url, timeout=10)
        data = response.json()

        targets = data.get("targets", [])
        if not targets:
            return []

        chembl_id = targets[0].get("target_chembl_id", "")
        if not chembl_id:
            return []

        print(f"   ✓ ChEMBL ID: {chembl_id}")

        safety_url = f"https://www.ebi.ac.uk/chembl/api/data/target/{chembl_id}.json"
        safety_resp = requests.get(safety_url, timeout=10)
        safety_data = safety_resp.json()

        side_effects = []
        cross_refs = safety_data.get("cross_references", [])
        for ref in cross_refs:
            if "tox" in ref.get("xref_src", "").lower():
                side_effects.append(ref.get("xref_name", "Unknown toxicity"))

        if side_effects:
            print(f"   ✓ Found {len(side_effects)} documented side effects")

        return side_effects

    except Exception:
        return []


# ─────────────────────────────────────────────
# MODULE 4: HISTORICAL COMPARISON
# Compares against known failed drug targets
# ─────────────────────────────────────────────

def compare_historical(gene_name: str, protein_name: str, protein_input: str) -> dict:
    """
    Checks if this protein or similar proteins
    have caused drug failures historically

    FIXED: Now checks gene_name, protein_name, AND original input
    so it doesn't miss matches when gene lookup fails
    """
    print(f"\n📚 Checking historical drug failure database...")

    result = {
        "found_in_history": False,
        "similar_targets": [],
        "historical_risk": "UNKNOWN",
        "notes": ""
    }

    # Build list of all name variants to check
    names_to_check = set()
    for name in [gene_name, protein_name, protein_input]:
        if name:
            names_to_check.add(name.upper().strip())

    # Direct match check across all name variants
    for check_name in names_to_check:
        if check_name in HISTORICAL_FAILED_TARGETS:
            entry = HISTORICAL_FAILED_TARGETS[check_name]
            result["found_in_history"] = True
            result["historical_risk"] = entry["risk"]
            result["notes"] = (
                f"This exact protein has caused drug failures: {entry['failure_reason']}"
            )
            print(f"   🚨 DIRECT MATCH found ({check_name}): {entry['failure_reason']}")
            return result

    # Family/similarity check
    for name in names_to_check:
        name_lower = name.lower()
        for family in HIGH_RISK_FAMILIES:
            if family in name_lower:
                if family not in result["similar_targets"]:
                    result["similar_targets"].append(family.upper())
                    result["historical_risk"] = "MODERATE"
                    result["notes"] = (
                        f"Belongs to high-risk protein family: {family.upper()}"
                    )
                    print(f"   ⚠ Similar to high-risk family: {family.upper()}")

    if not result["similar_targets"]:
        result["historical_risk"] = "LOW"
        result["notes"] = "No matches found in historical failure database"
        print(f"   ✓ No historical failure matches found")

    return result


# ─────────────────────────────────────────────
# MODULE 5: SIDE EFFECT PREDICTION
# Predicts likely side effects based on
# expression pattern
# ─────────────────────────────────────────────

def predict_side_effects(tissue_expression: dict, flagged_organs: list) -> list:
    """
    Predicts what side effects MIGHT occur
    based on where the protein is expressed
    """
    print(f"\n🔮 Predicting potential side effects...")

    organ_to_side_effect = {
        "heart":        "Cardiac arrhythmia or cardiotoxicity",
        "liver":        "Hepatotoxicity (liver damage)",
        "kidney":       "Nephrotoxicity (kidney damage)",
        "brain":        "Neurotoxicity or CNS side effects",
        "lung":         "Respiratory complications",
        "bone marrow":  "Hematotoxicity (blood cell production issues)",
        "spinal cord":  "Neurological side effects",
        "pancreas":     "Pancreatitis or glucose regulation issues",
        "intestine":    "Gastrointestinal disturbances",
        "stomach":      "Gastric ulcers or GI bleeding",
        "eye":          "Ocular toxicity",
        "skin":         "Dermatological reactions",
    }

    predictions = []

    for organ in flagged_organs:
        organ_lower = organ.lower()
        expression_level = tissue_expression.get(organ_lower, "Low")
        level_num = EXPRESSION_LEVELS.get(expression_level, 0)

        # Only predict for Medium or High expression
        if level_num >= 2 and organ_lower in organ_to_side_effect:
            side_effect = organ_to_side_effect[organ_lower]
            severity = "Severe" if level_num == 3 else "Moderate"
            predictions.append(
                f"{severity}: {side_effect} "
                f"(due to {expression_level} expression in {organ})"
            )

    if predictions:
        print(f"   ✓ Predicted {len(predictions)} potential side effects")
    else:
        print(f"   ✓ No major side effects predicted")

    return predictions


# ─────────────────────────────────────────────
# MODULE 6: RISK SCORING ENGINE
# ─────────────────────────────────────────────

def calculate_risk_score(
    tissue_expression: dict,
    known_side_effects: list,
    historical: dict,
    flagged_organs: list,
    target_tissue: Optional[str] = None
) -> tuple:
    """
    Calculates overall risk score (0-10) and confidence percentage
    Returns (risk_score, confidence, reasoning_steps)
    """

    score = 0.0
    reasoning_steps = []
    data_points_used = 0

    # ── Tissue Expression Scoring ──
    for organ, weight in VITAL_ORGANS.items():
        expression_level = tissue_expression.get(organ, "Not detected")
        level_num = EXPRESSION_LEVELS.get(expression_level, 0)

        if level_num > 0:
            organ_score = level_num * weight * 0.3
            data_points_used += 1

            # Reduce score if this is the intended target tissue
            if target_tissue and organ.lower() == target_tissue.lower():
                organ_score *= 0.2
                reasoning_steps.append(
                    f"  • {organ}: {expression_level} expression "
                    f"(TARGET tissue — risk reduced to +{organ_score:.1f})"
                )
            else:
                score += organ_score
                reasoning_steps.append(
                    f"  • {organ}: {expression_level} expression "
                    f"→ +{organ_score:.1f} risk points"
                )

    # ── Known Side Effects Scoring ──
    if known_side_effects:
        side_effect_score = len(known_side_effects) * 1.5
        score += side_effect_score
        data_points_used += 1
        reasoning_steps.append(
            f"  • {len(known_side_effects)} documented side effects "
            f"→ +{side_effect_score:.1f} risk points"
        )

    # ── Historical Comparison Scoring ──
    historical_scores = {
        "HIGH":    4.0,
        "MODERATE": 2.0,
        "LOW":     0.0,
        "UNKNOWN": 0.5
    }
    hist_score = historical_scores.get(historical["historical_risk"], 0)
    score += hist_score

    if hist_score > 0:
        data_points_used += 1
        reasoning_steps.append(
            f"  • Historical risk ({historical['historical_risk']}) "
            f"→ +{hist_score:.1f} risk points"
        )

    # Cap at 10
    score = min(score, 10.0)

    # ── Confidence Calculation ──
    max_data_points = 5
    base_confidence = min(data_points_used / max_data_points, 1.0)
    confidence = 40 + (base_confidence * 55)

    if not tissue_expression or len(tissue_expression) < 3:
        confidence *= 0.8
        reasoning_steps.append("  • Limited tissue data — confidence reduced")

    return round(score, 2), round(confidence, 1), reasoning_steps


# ─────────────────────────────────────────────
# MODULE 7: VERDICT GENERATOR
# ─────────────────────────────────────────────

def generate_verdict(risk_score: float) -> tuple:
    """Converts numeric score to traffic light verdict"""

    if risk_score >= 7.0:
        return (
            "HIGH RISK", "🔴",
            "This protein is NOT recommended as a drug target. "
            "High expression in vital organs poses serious toxicity risk. "
            "Consider alternative targets in the same pathway."
        )
    elif risk_score >= 4.0:
        return (
            "MODERATE RISK", "🟡",
            "This protein CAN be pursued with caution. "
            "Design drugs with high specificity to minimize off-target effects. "
            "Include organ function monitoring in clinical trials."
        )
    else:
        return (
            "LOW RISK", "🟢",
            "This protein is a GOOD drug target from a safety perspective. "
            "Expression pattern suggests limited off-target toxicity risk. "
            "Proceed to deeper druggability analysis."
        )


# ─────────────────────────────────────────────
# MAIN SAFETY AGENT
# ─────────────────────────────────────────────

def run_safety_agent(
    protein_input: str,
    disease_context: Optional[str] = None,
    target_tissue: Optional[str] = None
) -> SafetyReport:
    """
    MAIN FUNCTION — Run the complete Safety Agent

    Args:
        protein_input:    Protein name or UniProt ID (e.g. "EGFR" or "P00533")
        disease_context:  Disease being targeted (e.g. "lung cancer")
        target_tissue:    Primary tissue of interest (e.g. "lung")

    Returns:
        SafetyReport with complete analysis
    """

    print("\n" + "="*60)
    print("       SAFETY AGENT — LEVEL 3 ANALYSIS")
    print("="*60)
    print(f"  Protein Input  : {protein_input}")
    print(f"  Disease Context: {disease_context or 'Not specified'}")
    print(f"  Target Tissue  : {target_tissue or 'Not specified'}")
    print("="*60)

    # ── Step 1: Look up the protein ──
    protein_info = lookup_protein(protein_input)
    uniprot_id   = protein_info["uniprot_id"]
    protein_name = protein_info["protein_name"]
    gene_name    = protein_info["gene_name"]

    # ── Step 2: Get tissue expression ──
    time.sleep(0.5)
    tissue_expression = get_tissue_expression(gene_name, protein_input)

    # ── Step 3: Identify flagged vital organs ──
    flagged_organs = [
        organ for organ in VITAL_ORGANS
        if EXPRESSION_LEVELS.get(tissue_expression.get(organ, "Not detected"), 0) >= 2
    ]

    # ── Step 4: Get known side effects ──
    time.sleep(0.5)
    known_side_effects = get_known_side_effects(gene_name, protein_input)

    # ── Step 5: Historical comparison (FIXED — checks all name variants) ──
    historical = compare_historical(gene_name, protein_name, protein_input)

    # ── Step 6: Predict side effects ──
    predicted_side_effects = predict_side_effects(tissue_expression, flagged_organs)

    # ── Step 7: Calculate risk score ──
    risk_score, confidence, reasoning_steps = calculate_risk_score(
        tissue_expression,
        known_side_effects,
        historical,
        flagged_organs,
        target_tissue
    )

    # ── Step 8: Generate verdict ──
    verdict, emoji, recommendation = generate_verdict(risk_score)

    # ── Build full reasoning text ──
    reasoning = (
        f"REASONING BREAKDOWN:\n"
        f"{chr(10).join(reasoning_steps)}\n\n"
        f"HISTORICAL NOTE: {historical['notes']}\n\n"
        f"RECOMMENDATION: {recommendation}"
    )

    return SafetyReport(
        protein_id=uniprot_id,
        protein_name=protein_name,
        tissue_expression=tissue_expression,
        flagged_organs=flagged_organs,
        known_side_effects=known_side_effects,
        historical_comparison=historical,
        predicted_side_effects=predicted_side_effects,
        risk_score=risk_score,
        confidence=confidence,
        verdict=verdict,
        reasoning=reasoning
    )


# ─────────────────────────────────────────────
# REPORT PRINTER
# ─────────────────────────────────────────────

def print_report(report: SafetyReport):
    """Prints the safety report in a clean readable format"""

    verdict_emoji = {
        "HIGH RISK":     "🔴",
        "MODERATE RISK": "🟡",
        "LOW RISK":      "🟢"
    }
    emoji = verdict_emoji.get(report.verdict, "⚪")

    print("\n" + "="*60)
    print("           SAFETY AGENT — FINAL REPORT")
    print("="*60)
    print(f"  Protein       : {report.protein_name}")
    print(f"  UniProt ID    : {report.protein_id}")
    print(f"  Risk Score    : {report.risk_score} / 10")
    print(f"  Confidence    : {report.confidence}%")
    print(f"  VERDICT       : {emoji} {report.verdict}")
    print("="*60)

    print("\n📍 TISSUE EXPRESSION:")
    if report.tissue_expression:
        for tissue, level in list(report.tissue_expression.items())[:10]:
            bar = "█" * EXPRESSION_LEVELS.get(level, 0)
            print(f"   {tissue:<22} {level:<15} {bar}")
    else:
        print("   No expression data available")

    print("\n🚨 FLAGGED VITAL ORGANS:")
    if report.flagged_organs:
        for organ in report.flagged_organs:
            level = report.tissue_expression.get(organ, "")
            print(f"   ⚠  {organ} ({level} expression)")
    else:
        print("   ✓ No vital organs flagged")

    print("\n📋 KNOWN SIDE EFFECTS:")
    if report.known_side_effects:
        for effect in report.known_side_effects:
            print(f"   • {effect}")
    else:
        print("   ✓ No documented side effects found")

    print("\n🔮 PREDICTED SIDE EFFECTS:")
    if report.predicted_side_effects:
        for pred in report.predicted_side_effects:
            print(f"   • {pred}")
    else:
        print("   ✓ No major side effects predicted")

    print("\n📚 HISTORICAL COMPARISON:")
    print(f"   Risk Level : {report.historical_comparison['historical_risk']}")
    print(f"   Notes      : {report.historical_comparison['notes']}")

    print(f"\n💡 REASONING:\n{report.reasoning}")
    print("\n" + "="*60)


def save_report_json(report: SafetyReport, filepath: str):
    """Saves report as a JSON file"""
    import os
    os.makedirs(os.path.dirname(filepath) if os.path.dirname(filepath) else ".", exist_ok=True)

    data = {
        "protein_id":            report.protein_id,
        "protein_name":          report.protein_name,
        "tissue_expression":     report.tissue_expression,
        "flagged_organs":        report.flagged_organs,
        "known_side_effects":    report.known_side_effects,
        "historical_comparison": report.historical_comparison,
        "predicted_side_effects":report.predicted_side_effects,
        "risk_score":            report.risk_score,
        "confidence":            report.confidence,
        "verdict":               report.verdict,
        "reasoning":             report.reasoning
    }

    with open(filepath, "w") as f:
        json.dump(data, f, indent=2)

    print(f"\n💾 Report saved to: {filepath}")


# ─────────────────────────────────────────────
# RUN THE AGENT
# Change the protein names below to test others
# ─────────────────────────────────────────────

if __name__ == "__main__":

    # Change this path to wherever you want to save reports
    # For Kaggle: "/kaggle/working/"
    # For local:  "./"
    OUTPUT_DIR = "/kaggle/working/"

    # ── TEST 1: EGFR — Good lung cancer target ──
    print("\n\n🧪 TEST 1: EGFR — Known lung cancer drug target")
    report1 = run_safety_agent(
        protein_input="EGFR",
        disease_context="lung cancer",
        target_tissue="lung"
    )
    print_report(report1)
    save_report_json(report1, OUTPUT_DIR + "EGFR_safety_report.json")

    # ── TEST 2: KCNH2 (hERG) — Known dangerous target ──
    print("\n\n🧪 TEST 2: KCNH2 (hERG) — Known dangerous cardiac target")
    report2 = run_safety_agent(
        protein_input="KCNH2",
        disease_context="cancer",
        target_tissue="tumor"
    )
    print_report(report2)
    save_report_json(report2, OUTPUT_DIR + "hERG_safety_report.json")

    # ── TEST 3: TP53 — Tumor suppressor ──
    print("\n\n🧪 TEST 3: TP53 — Tumor suppressor protein")
    report3 = run_safety_agent(
        protein_input="TP53",
        disease_context="colon cancer",
        target_tissue="colon"
    )
    print_report(report3)
    save_report_json(report3, OUTPUT_DIR + "TP53_safety_report.json")



🧪 TEST 1: EGFR — Known lung cancer drug target

       SAFETY AGENT — LEVEL 3 ANALYSIS
  Protein Input  : EGFR
  Disease Context: lung cancer
  Target Tissue  : lung

🔍 Looking up protein: EGFR
   ✓ Found: Receptor protein-tyrosine kinase | UniProt: E7BSV0 | Gene: EGFR

🧬 Checking tissue expression for: EGFR
   ℹ Using built-in expression database
   ✓ Found built-in data for: EGFR

⚠️  Checking known side effects for: EGFR
   ✓ ChEMBL ID: CHEMBL203
   ✓ ChEMBL ID: CHEMBL203
   ℹ No documented side effects found in ChEMBL

📚 Checking historical drug failure database...
   ✓ No historical failure matches found

🔮 Predicting potential side effects...
   ✓ Predicted 2 potential side effects

           SAFETY AGENT — FINAL REPORT
  Protein       : Receptor protein-tyrosine kinase
  UniProt ID    : E7BSV0
  Risk Score    : 3.0 / 10
  Confidence    : 84.0%
  VERDICT       : 🟢 LOW RISK

📍 TISSUE EXPRESSION:
   lung                   High            ███
   liver                  Medium     

In [16]:
import requests
import rdkit
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# -------------------------------
# 1. PDB → UniProt mapping
# -------------------------------
def pdb_to_uniprot(pdb_id):
    url = f"https://www.ebi.ac.uk/pdbe/api/mappings/uniprot/{pdb_id.lower()}"
    response = requests.get(url)

    if response.status_code != 200:
        return None

    data = response.json()

    try:
        uniprot_ids = list(data[pdb_id.lower()]["UniProt"].keys())
        return uniprot_ids[0] if uniprot_ids else None
    except:
        return None


# -------------------------------
# 2. UniProt → ChEMBL mapping
# -------------------------------
def uniprot_to_chembl(uniprot_id):
    url = f"https://www.ebi.ac.uk/chembl/api/data/target.json?target_components__accession={uniprot_id}"
    response = requests.get(url)

    if response.status_code != 200:
        return None

    data = response.json()

    try:
        return data["targets"][0]["target_chembl_id"]
    except:
        return None


# -------------------------------
# 3. Fetch ligands
# -------------------------------
def fetch_ligands(target_chembl_id, limit=100):
    url = f"https://www.ebi.ac.uk/chembl/api/data/activity.json?target_chembl_id={target_chembl_id}&limit={limit}"
    response = requests.get(url)

    if response.status_code != 200:
        return []

    data = response.json()
    ligands = []

    for item in data['activities']:
        smi = item.get('canonical_smiles')
        ic50 = item.get('standard_value')
        unit = item.get('standard_units')

        if smi and ic50:
            try:
                ligands.append({
                    "smiles": smi,
                    "ic50": float(ic50),
                    "unit": unit
                })
            except:
                continue

    return ligands


# -------------------------------
# 4. Filter active ligands
# -------------------------------
def filter_active(ligands):
    return [l for l in ligands if l["unit"] == "nM" and l["ic50"] < 1000]


# -------------------------------
# 5. Drug-likeness
# -------------------------------
def drug_properties(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 0, 4

    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    h_donors = Descriptors.NumHDonors(mol)
    h_acceptors = Descriptors.NumHAcceptors(mol)

    violations = sum([
        mw >= 500,
        logp >= 5,
        h_donors > 5,
        h_acceptors > 10
    ])

    return (4 - violations) / 4, violations


# -------------------------------
# 6. Synthetic accessibility
# -------------------------------
def synthetic_accessibility(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 0

    complexity = Descriptors.BertzCT(mol)

    if complexity < 500:
        return 0.9
    elif complexity < 1000:
        return 0.7
    elif complexity < 1500:
        return 0.5
    else:
        return 0.3


# -------------------------------
# 7. Binding score
# -------------------------------
def binding_score(ic50):
    if ic50 < 100:
        return 1.0
    elif ic50 < 500:
        return 0.8
    elif ic50 < 1000:
        return 0.6
    else:
        return 0.3


# -------------------------------
# 8. Chemistry Agent (FINAL)
# -------------------------------
def chemistry_agent_from_pdb(pdb_id):
    print(f"\n🔍 Processing PDB ID: {pdb_id}")

    uniprot = pdb_to_uniprot(pdb_id)
    if not uniprot:
        return {"error": "Failed to map PDB → UniProt"}

    chembl_id = uniprot_to_chembl(uniprot)
    if not chembl_id:
        return {"error": "Failed to map UniProt → ChEMBL"}

    ligands = fetch_ligands(chembl_id)
    active = filter_active(ligands)

    if not active:
        return {"error": "No active ligands found"}

    results = []

    for l in active:
        dl, violations = drug_properties(l["smiles"])
        sa = synthetic_accessibility(l["smiles"])
        bs = binding_score(l["ic50"])

        score = 0.4 * dl + 0.3 * sa + 0.3 * bs

        results.append({
            "smiles": l["smiles"],
            "ic50": l["ic50"],
            "drug_likeness": round(dl, 3),
            "violations": violations,
            "synthetic_accessibility": round(sa, 3),
            "binding_score": round(bs, 3),
            "score": round(score, 3)
        })

    df = pd.DataFrame(results)

    chemistry_score = df["score"].mean()
    confidence = min(1.0, len(active) / 50)

    verdict = (
        "HIGH feasibility" if chemistry_score > 0.75 else
        "MODERATE feasibility" if chemistry_score > 0.5 else
        "LOW feasibility"
    )

    return {
        "pdb_id": pdb_id,
        "uniprot": uniprot,
        "chembl_id": chembl_id,
        "active_ligands": len(active),
        "chemistry_score": round(chemistry_score, 3),
        "confidence": round(confidence, 3),
        "verdict": verdict,
        "sample_results": df.head(5).to_dict(orient="records")
    }


# -------------------------------
# 9. Run Example
# -------------------------------
if __name__ == "__main__":
    result = chemistry_agent_from_pdb("1UWH")

    print("\n=== FINAL OUTPUT ===")
    for k, v in result.items():
        print(f"{k}: {v}")


🔍 Processing PDB ID: 1UWH

=== FINAL OUTPUT ===
pdb_id: 1UWH
uniprot: P15056
chembl_id: CHEMBL5145
active_ligands: 46
chemistry_score: 0.831
confidence: 0.92
verdict: HIGH feasibility
sample_results: [{'smiles': 'CC(C)(C)c1nc(-c2ccc(F)cc2)c(-c2ccncc2)[nH]1', 'ic50': 900.0, 'drug_likeness': 1.0, 'violations': 0, 'synthetic_accessibility': 0.7, 'binding_score': 0.6, 'score': 0.79}, {'smiles': 'Oc1ccc(-c2[nH]c(-c3ccccc3)nc2-c2ccncc2)cc1', 'ic50': 339.0, 'drug_likeness': 1.0, 'violations': 0, 'synthetic_accessibility': 0.7, 'binding_score': 0.8, 'score': 0.85}, {'smiles': 'CC(C)(CNC(=O)Nc1ccc(Cl)cc1)c1nc(-c2ccc(Cl)c(O)c2)c(-c2ccncc2)[nH]1', 'ic50': 13.0, 'drug_likeness': 0.75, 'violations': 1, 'synthetic_accessibility': 0.5, 'binding_score': 1.0, 'score': 0.75}, {'smiles': 'CC(C)(CNS(C)(=O)=O)c1nc(-c2ccc(Cl)c(O)c2)c(-c2ccncc2)[nH]1', 'ic50': 14.0, 'drug_likeness': 1.0, 'violations': 0, 'synthetic_accessibility': 0.5, 'binding_score': 1.0, 'score': 0.85}, {'smiles': 'CN(C)CCCNC(=O)c1nc(-c2

In [17]:
# ── Inline imports so this cell is self-contained ──
import numpy as np
import os
import requests
import py3Dmol
from Bio.PDB import PDBParser, PDBList
from scipy.spatial import KDTree
from sklearn.cluster import DBSCAN

HYDROPHOBIC = {"ALA","VAL","LEU","ILE","MET","PHE","TRP","PRO"}

class StructureAgent:

    def __init__(self, protein_input):
        self.pdb_file = self.resolve_input(protein_input)
        self.models = []
        self.load_structure()

    # ------------------------------------------------
    # Resolve input (Protein name / PDB ID / File)
    # ------------------------------------------------
    def resolve_input(self, protein_input):

        protein_input = protein_input.strip()

        if os.path.exists(protein_input):
            return protein_input

        clean = protein_input.upper()

        if "PDB" in clean:
            clean = clean.split()[-1]

        if len(clean) == 4 and clean.isalnum():
            return self.download_pdb(clean)

        # Text search fallback
        print("Searching RCSB for:", protein_input)

        query = {
            "query": {
                "type": "terminal",
                "service": "text",
                "parameters": {"value": protein_input}
            },
            "return_type": "entry",
            "request_options": {"pager": {"start": 0, "rows": 1}}
        }

        response = requests.post(
            "https://search.rcsb.org/rcsbsearch/v1/query",
            json=query
        )

        data = response.json()

        if "result_set" not in data or len(data["result_set"]) == 0:
            raise ValueError("No structure found. Try PDB ID directly.")

        pdb_id = data["result_set"][0]["identifier"]
        print("Found PDB:", pdb_id)

        return self.download_pdb(pdb_id)

    def download_pdb(self, pdb_id):

        pdbl = PDBList()
        file_path = pdbl.retrieve_pdb_file(
            pdb_id,
            pdir=".",
            file_format="pdb"
        )

        new_name = f"{pdb_id}.pdb"

        if os.path.exists(new_name):
            return new_name

        os.rename(file_path, new_name)

        return new_name

    # ------------------------------------------------
    # Load Structure
    # ------------------------------------------------
    def load_structure(self):

        parser = PDBParser(QUIET=True)
        structure = parser.get_structure("protein", self.pdb_file)

        for model in structure:

            atoms = []
            residues = []
            res_ids = []

            for chain in model:
                prev_res = None

                for residue in chain:

                    if residue.get_resname() == "HOH":
                        continue

                    res_id = residue.get_id()[1]

                    if prev_res and res_id != prev_res + 1:
                        print(f"GAP detected between {prev_res} and {res_id}")

                    prev_res = res_id

                    for atom in residue:
                        if atom.get_parent().id[0] != " ":
                            continue
                        atoms.append(atom.coord)
                        residues.append(residue.get_resname())
                        res_ids.append(res_id)

            self.models.append({
                "atoms": np.array(atoms),
                "residues": residues,
                "res_ids": res_ids
            })

    # ------------------------------------------------
    # Pocket Detection (Improved)
    # ------------------------------------------------
    def detect_pocket(self, atoms):

        tree = KDTree(atoms)
        spacing = 1.0

        min_c = np.min(atoms, axis=0) - 2
        max_c = np.max(atoms, axis=0) + 2

        x = np.arange(min_c[0], max_c[0], spacing)
        y = np.arange(min_c[1], max_c[1], spacing)
        z = np.arange(min_c[2], max_c[2], spacing)

        grid = np.array(np.meshgrid(x, y, z)).T.reshape(-1, 3)

        cavity = []

        for point in grid:
            dist, _ = tree.query(point)

            if dist > 2.8:
                neighbors = tree.query_ball_point(point, r=5.0)
                if 8 < len(neighbors) < 80:
                    cavity.append(point)

        cavity = np.array(cavity)

        if len(cavity) == 0:
            return None

        clustering = DBSCAN(eps=2.5, min_samples=8).fit(cavity)
        labels = clustering.labels_

        largest = None
        max_size = 0

        for label in set(labels):
            if label == -1:
                continue
            cluster = cavity[labels == label]
            if len(cluster) > max_size:
                largest = cluster
                max_size = len(cluster)

        return largest

    # ------------------------------------------------
    # Extract Pocket Residues
    # ------------------------------------------------
    def extract_pocket_residues(self, atoms, residues, res_ids, pocket):

        tree = KDTree(atoms)
        pocket_res = set()

        for point in pocket:
            neighbors = tree.query_ball_point(point, r=4.0)
            for idx in neighbors:
                pocket_res.add(res_ids[idx])

        return sorted(pocket_res)

    # ------------------------------------------------
    # Evaluate + Visualize
    # ------------------------------------------------
    def evaluate(self):

        print("Total Models:", len(self.models))

        model = self.models[0]

        atoms = model["atoms"]
        residues = model["residues"]
        res_ids = model["res_ids"]

        pocket = self.detect_pocket(atoms)

        if pocket is None:
            print("No druggable pocket detected.")
            return

        pocket_res_ids = self.extract_pocket_residues(
            atoms, residues, res_ids, pocket
        )

        volume = len(pocket)
        depth = np.max(pocket[:,2]) - np.min(pocket[:,2])

        print("Pocket Volume:", volume)
        print("Pocket Depth:", round(depth,2))
        print("Pocket Residue IDs:", pocket_res_ids)

        self.visualize(pocket_res_ids)

    # ------------------------------------------------
    # AlphaFold-style Visualization
    # ------------------------------------------------
    def visualize(self, pocket_res_ids):

        with open(self.pdb_file, "r") as f:
            pdb_data = f.read()

        view = py3Dmol.view(width=900, height=700)
        view.addModel(pdb_data, "pdb")

        # Cartoon structure
        view.setStyle({"cartoon": {"color": "spectrum"}})

        # Highlight pocket residues
        if pocket_res_ids:
            view.setStyle(
                {"resi": pocket_res_ids},
                {"stick": {"colorscheme": "redCarbon"}}
            )

        view.addSurface(py3Dmol.VDW, {"opacity": 0.2})
        view.zoomTo()
        view.spin(True)

        # Save as HTML for Kaggle (no interactive display support)
        html_path = self.pdb_file.replace(".pdb", "_pocket.html")
        view.write_html(html_path)
        print(f"3D visualization saved to: {html_path}")


# ── Biology helpers (inline) ──
OPEN_TARGETS_URL = "https://api.platform.opentargets.org/api/v4/graphql"

def get_ensembl_id(gene_symbol):
    query = """
    query SearchTarget($symbol: String!) {
      search(queryString: $symbol, entityNames: ["target"]) {
        hits {
          object {
            ... on Target { id approvedSymbol approvedName }
          }
        }
      }
    }
    """
    response = requests.post(
        OPEN_TARGETS_URL,
        json={"query": query, "variables": {"symbol": gene_symbol}},
        timeout=15
    )
    data = response.json()
    hits = data.get("data", {}).get("search", {}).get("hits", [])
    for hit in hits:
        obj = hit.get("object", {})
        if obj.get("approvedSymbol", "").upper() == gene_symbol.upper():
            return obj["id"], obj["approvedName"]
    if hits:
        obj = hits[0].get("object", {})
        return obj.get("id"), obj.get("approvedName")
    return None, None

def get_disease_associations(ensembl_id):
    query = """
    query GetAssociations($id: String!) {
      target(ensemblId: $id) {
        associatedDiseases(page: {index: 0, size: 200}) {
          rows { score disease { id name } datatypeScores { id score } }
        }
      }
    }
    """
    response = requests.post(
        OPEN_TARGETS_URL,
        json={"query": query, "variables": {"id": ensembl_id}},
        timeout=15
    )
    data = response.json()
    return data.get("data", {}).get("target", {}).get("associatedDiseases", {}).get("rows", [])

def find_disease_match(associations, disease_query):
    disease_query_lower = disease_query.lower().strip()
    best_match, best_score = None, 0
    for row in associations:
        name = row["disease"]["name"].lower().strip()
        if disease_query_lower in name or name in disease_query_lower:
            if row["score"] > best_score:
                best_score = row["score"]
                best_match = row
    return best_match

def parse_evidence_types(datatype_scores):
    evidence_map = {
        "genetic_association": "Genetic mutations in patients",
        "somatic_mutation":    "Somatic mutations in disease tissue",
        "known_drug":          "Existing drugs target this protein",
        "affected_pathway":    "Protein is in disease pathway",
        "literature":          "Mentioned in research papers",
        "animal_model":        "Animal knockout/overexpression studies",
        "rna_expression":      "Gene expression changes in disease",
        "text_mining":         "Text mining from publications"
    }
    return {evidence_map.get(item["id"], item["id"]): round(item["score"], 3)
            for item in datatype_scores}

def rule_based_assessment(overall_score, evidence):
    flags, avoid_flags = [], []
    if evidence.get("Genetic mutations in patients", 0) > 0.3:
        flags.append("Strong genetic evidence: mutations found in patients")
    if evidence.get("Somatic mutations in disease tissue", 0) > 0.3:
        flags.append("Somatic mutations present - likely driver not bystander")
    if evidence.get("Existing drugs target this protein", 0) > 0.2:
        flags.append("Already a known drug target - druggability validated")
    if evidence.get("Animal knockout/overexpression studies", 0) > 0.2:
        flags.append("Animal model evidence: blocking this protein affects disease")
    if (evidence.get("Genetic mutations in patients", 0) < 0.1 and
            evidence.get("Somatic mutations in disease tissue", 0) < 0.1):
        avoid_flags.append("No genetic/somatic mutation evidence - may be bystander")
    if (evidence.get("Gene expression changes in disease", 0) > 0.3 and
            evidence.get("Genetic mutations in patients", 0) < 0.1):
        avoid_flags.append("Only expression change - likely bystander, not driver")
    if overall_score >= 0.6:
        verdict = "STRONG TARGET"
    elif overall_score >= 0.35:
        verdict = "MODERATE TARGET"
    elif overall_score >= 0.15:
        verdict = "WEAK TARGET - needs more evidence"
    else:
        verdict = "AVOID - insufficient disease link"
    return {"verdict": verdict, "overall_score": round(overall_score, 3),
            "positive_signals": flags, "warning_signals": avoid_flags}


# ── Decision Agent ──────────────────────────────────────────────────

class DecisionAgent:

    def __init__(self, gene, disease, pdb_input):
        self.gene = gene
        self.disease = disease
        self.pdb_input = pdb_input

    def run_biology(self):
        ensembl_id, name = get_ensembl_id(self.gene)
        if not ensembl_id:
            return None
        associations = get_disease_associations(ensembl_id)
        match = find_disease_match(associations, self.disease)
        if not match:
            return {"verdict": "NO_MATCH", "score": 0}
        evidence = parse_evidence_types(match.get("datatypeScores", []))
        assessment = rule_based_assessment(match["score"], evidence)
        return {"score": match["score"], "verdict": assessment["verdict"], "evidence": evidence}

    def run_structure(self):
        agent = StructureAgent(self.pdb_input)
        model = agent.models[0]
        pocket = agent.detect_pocket(model["atoms"])
        if pocket is None:
            return {"druggable": False, "volume": 0}
        volume = len(pocket)
        return {"druggable": volume > 100, "volume": volume}

    def decide(self):
        bio = self.run_biology()
        struct = self.run_structure()
        if bio is None:
            return "REJECT: No biological data"
        bio_score = bio["score"]
        strong_bio = bio_score >= 0.5
        druggable = struct["druggable"]
        if strong_bio and druggable:
            verdict = "PRIORITIZE TARGET"
        elif strong_bio and not druggable:
            verdict = "HARD TARGET (BIOLOGY GOOD, STRUCTURE BAD)"
        elif not strong_bio and druggable:
            verdict = "IRRELEVANT TARGET (STRUCTURE OK, BIOLOGY WEAK)"
        else:
            verdict = "REJECT TARGET"
        return {
            "biology_score": bio_score,
            "structure_volume": struct["volume"],
            "final_decision": verdict
        }


# ── Run ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    agent = DecisionAgent(gene="BRAF", disease="melanoma", pdb_input="1UWH")
    result = agent.decide()
    print("\nFINAL DECISION")
    print(result)


GAP detected between 599 and 612
GAP detected between 722 and 1723
GAP detected between 599 and 612
GAP detected between 722 and 1723

FINAL DECISION
{'biology_score': 0.8185527839444029, 'structure_volume': 8487, 'final_decision': 'PRIORITIZE TARGET'}
